In [1]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize

In [2]:
sessions = {
    "s1": ["t1", "t2", "t2"],
    "s2": ["t1", "t3"],
    "s3": ["t2", "t3", "t3"],
}

event_texts = {
    "t1": "receiving block",
    "t2": "replication started",
    "t3": "block verification failed",
}

session_ids = list(sessions)
documents = [" ".join(sessions[sid]) for sid in session_ids]

In [4]:
### 1. Contagens

count_vectorizer = CountVectorizer(
    token_pattern=r"(?u)\b\w+\b"
)

X_counts = count_vectorizer.fit_transform(documents).toarray()

print(count_vectorizer.get_feature_names_out())
print("Cada coluna representa um template, e cada valor indica quantas vezes ele ocorreu na sessão.")
print(X_counts)
print(X_counts.shape)

['t1' 't2' 't3']
Cada coluna representa um template, e cada valor indica quantas vezes ele ocorreu na sessão.
[[1 2 0]
 [1 0 1]
 [0 1 2]]
(3, 3)


In [ ]:
### 2. TF-IDF

tfidf_vectorizer = TfidfVectorizer(
    token_pattern=r"(?u)\b\w+\b"
)

X_tfidf = tfidf_vectorizer.fit_transform(documents).toarray()

print(tfidf_vectorizer.get_feature_names_out())
print("Eventos frequentes no corpus recebem menor peso; eventos mais específicos recebem maior peso.")
print(X_tfidf)
print(X_tfidf.shape)

['t1' 't2' 't3']
[[0.4472136  0.89442719 0.        ]
 [0.70710678 0.         0.70710678]
 [0.         0.4472136  0.89442719]]
(3, 3)


In [9]:
### 3. Embeddings
from pathlib import Path
from sentence_transformers import SentenceTransformer

ROOT = Path.cwd()

while not (
    ROOT / "artifacts/models/huggingface"
).exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

MODEL_PATH = (
    ROOT
    / "artifacts/models/huggingface"
    / "models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2"
    / "snapshots"
    / "e8f8c211226b894fcb81acc59f3b34ba3efd5f42"
)

assert MODEL_PATH.exists(), MODEL_PATH

model = SentenceTransformer(
    str(MODEL_PATH),
    model_kwargs={"local_files_only": True},
)

print(model.get_embedding_dimension())

# Codifica cada tipo de evento uma única vez
template_ids = list(event_texts)
template_texts = [event_texts[t] for t in template_ids]

template_embeddings = model.encode(
    template_texts,
    normalize_embeddings=True
)

embedding_by_template = dict(
    zip(template_ids, template_embeddings)
)

# Agrega os embeddings dos eventos de cada sessão pela média
X_semantic = []

for session_id in session_ids:
    vectors = [
        embedding_by_template[event]
        for event in sessions[session_id]
    ]
    session_vector = np.mean(vectors, axis=0)
    session_vector = normalize(
        session_vector.reshape(1, -1)
    )[0]
    X_semantic.append(session_vector)

X_semantic = np.asarray(X_semantic, dtype=np.float32)

print(X_semantic.shape)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10181.22it/s]


384
(3, 384)


In [13]:
# representação híbrida:

X_hybrid = np.concatenate(
    [X_semantic, X_counts.astype(np.float32)],
    axis=1
)

print(X_counts)
print(X_hybrid)

[[1 2 0]
 [1 0 1]
 [0 1 2]]
[[-0.04882156  0.00376994 -0.08721495 ...  1.          2.
   0.        ]
 [-0.06187575  0.04238345 -0.07344615 ...  1.          0.
   1.        ]
 [-0.07663456  0.02298535 -0.06911753 ...  0.          1.
   2.        ]]
